In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
from skimage import morphology, segmentation
from scipy.ndimage import rotate
from skimage import segmentation, morphology
from matplotlib.colors import hsv_to_rgb
import Chain_Analysis_Functions as caf

## Step 1: Read in the Image and create the initial mask

In [ ]:
original_image_name = '55-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.2_0mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask = caf.create_binary_mask(image_path, brightness=1.0, contrast=1.0, saturation=1.0,
                           temperature=0, R_min=0, G_min=0, B_min=30, V_min=0.1,
                           method="adaptive", adaptive_block_size=10, adaptive_offset=0.01)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 8))

ax1.imshow(mask, cmap='gray')
ax1.set_title("Original Image")
ax1.axis("off")

# Top-left corner (works as-is)
ax2.imshow(mask[40:520, 7000:8000], cmap='gray')
ax2.set_title("Top left corner")
ax2.axis("off")

# Bottom-right corner — corrected slicing
ax3.imshow(mask[-520:, -520:], cmap='gray')
ax3.set_title("Bottom right corner")
ax3.axis("off")

plt.tight_layout()
plt.show()

## Step 2: Rotate the mask such that the particles align into perfect rows and columns

### Perform Rough Rotation

In [ ]:
##  This time i just did the full rotation
angle = 0.1765
rot_mask = rotate(mask, angle, reshape=False, order=0, mode='constant', cval=0)
rot_mask = morphology.remove_small_objects(rot_mask[-400:,0:9000], min_size= 100) #Remove noise to keep it from interfering
caf.display_mask(rot_mask, 20, 5)

#### Identify the appropriate rotation angle

In [ ]:
rot_angle_mask = morphology.remove_small_objects(rot_mask, min_size= 200) #Remove noise to keep it from interfering
rot_angle_mask, angle = caf.rotate_mask_until_balanced(rot_angle_mask, angle_step=0.00001, tolerance=0.00001, min_step=0.00001, max_angle=2, search_rows = 100, search_columns = 3000)
print(angle)

In [ ]:
angle = 0.1765

#### Apply the rotation to the actual mask and confirm

In [ ]:
mask_rotated = rotate(mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(mask_rotated)

### Crop the mask if necessary

In [ ]:
mask_new= mask_rotated[50:,:]
caf.display_mask(mask_new[0:500,:],20,5)

#### Update the mask

In [ ]:
mask = mask_new
del mask_new

## Step 3: Identify the reference particle bounding box

In [ ]:
dy = 6500
minr  = -5517
maxr = -5217
minc = 6811
maxc = 7111
ref_bbox = (minr,minc,maxr,maxc)
caf.show_reference(mask, ref_bbox)

## Step 4: Ensure reference crop is filled properly with set parameters

### Perform Morphological Operations manually

In [ ]:
#Define the minimum and maximum row indices used to crop the reference particle from the mask.
minr  = -5517

# Define the ending row index used to crop the reference particle from the mask.
maxr = -5217

# Define the minimum column index used to crop the reference particle from the mask.
minc = 6811

# Define the ending column index used to crop the reference particle from the mask.
maxc = 7111

# Extract the selected rectangular region from the full mask using the specified row and column bounds.
ref_crop0 = mask[minr:maxr,minc:maxc]

# Create an independent copy of the cropped reference mask so that the original crop remains unchanged.
ref_crop = ref_crop0.copy()

# Display the initially cropped reference mask for visual inspection.
caf.display_mask(ref_crop,6,6)

# Remove a small region from the lower-left portion of the reference mask.
ref_crop[-40:-32,:8]  = 0

# Display the reference mask after manually removing the unwanted region.
caf.display_mask(ref_crop,6,6)

# Define the minimum connected-component size to retain during small-object removal.
min_size= 40

# Remove connected components smaller than the specified minimum size from the reference mask.
ref_crop = morphology.remove_small_objects(ref_crop, min_size=min_size)

# Display the reference mask after removing small objects.
caf.display_mask(ref_crop,6,6)

# Fill horizontal gaps within the lower portion of the reference mask, allowing gaps up to 40 pixels wide.
ref_crop[-31:,:] = caf.fill_column_gaps(ref_crop[-31:,:], max_gap=40)

# Fill horizontal gaps within the upper portion of the reference mask, allowing gaps up to 40 pixels wide.
ref_crop[:31,:] = caf.fill_column_gaps(ref_crop[:31,:], max_gap=40)

# Fill horizontal gaps within the central columns of the reference mask, allowing gaps up to 60 pixels wide.
ref_crop[:,30:-30] = caf.fill_column_gaps(ref_crop[:,30:-30], max_gap=60)

# Display the reference mask after filling the specified gaps.
caf.display_mask(ref_crop, 6, 6)

# Apply a binary morphological opening using a disk-shaped structuring element with radius 2.
ref_crop = binary_opening(ref_crop, structure=disk(2))

# Display the reference mask after morphological opening.
caf.display_mask(ref_crop,6,6)

# Remove an unwanted region from the lower-left portion of the reference mask.
ref_crop[-80:-60,:31] = 0

# Remove an unwanted region from the lower-right portion of the reference mask.
ref_crop[-80:-50,-23:] = 0

# Remove an unwanted region from the right side of the central portion of the reference mask.
ref_crop[60:120,-30:] = 0

# Display the reference mask after the additional manual removals.
caf.display_mask(ref_crop,6,6)

# Define the number of pixels to add as padding around the reference mask.
pad = 100

# Add 100 pixels of zero-valued padding around all sides of the reference mask.
ref_crop = np.pad(ref_crop, pad_width=pad, mode='constant')

# Display the padded reference mask before morphological closing.
caf.display_mask(ref_crop, 6, 6)

# Apply binary morphological closing using a disk-shaped structuring element with radius 15
# to close larger gaps and smooth the reference mask.
ref_crop = binary_closing(ref_crop, structure=disk(15))

# Display the reference mask after morphological closing.
caf.display_mask(ref_crop,6,6)

# Remove the padding that was added around the reference mask, restoring its original dimensions.
ref_crop = ref_crop[pad:-pad, pad:-pad]

# Display the final processed reference mask after removing the padding.
caf.display_mask(ref_crop, 6, 6)

# Convert the binary reference mask into a three-channel floating-point image
# so that a colored overlay can be created.
overlay_img = np.stack([ref_crop] * 3, axis=-1).astype(float)

# Extract the original mask region using the same row and column bounds as the reference crop.
cropmask = mask[minr:maxr, minc:maxc]

# Set the red channel to full intensity wherever the original cropped mask contains data.
overlay_img[cropmask.astype(bool), 0] = 1.0  # Red

# Set the green channel to zero wherever the original cropped mask contains data.
overlay_img[cropmask.astype(bool), 1] = 0.0

# Set the blue channel to zero wherever the original cropped mask contains data.
overlay_img[cropmask.astype(bool), 2] = 0.0

# Create a new figure for displaying the overlay.
plt.figure(figsize=(6, 6))

# Display the processed reference mask with the original cropped mask overlaid in red.
plt.imshow(overlay_img)

# Set the title describing the overlay visualization.
plt.title("Overlay Mask in Red")

# Remove the axes from the displayed image.
plt.axis('off')

# Display the final overlay visualization.
plt.show()


### Overlay on First Particle

In [ ]:
minr = 31
maxr = 331
minc = 44
maxc = 344
overlay_img = np.stack([ref_crop] * 3, axis=-1).astype(float)
cropmask = mask[minr:maxr, minc:maxc]
caf.display_mask(cropmask,6,6)
overlay_img[cropmask.astype(bool), 0] = 1.0  # Red
overlay_img[cropmask.astype(bool), 1] = 0.0
overlay_img[cropmask.astype(bool), 2] = 0.0
plt.figure(figsize=(6, 6))
plt.imshow(overlay_img)
plt.title("Overlay Mask in Red")
plt.axis('off')
plt.show()

## Step 5: Ensure Last Data is Properly Identified

In [ ]:
caf.display_mask(mask[:500,-500:10311],10,10)

In [ ]:
last_x, last_y = caf.check_last_row_and_column(mask, min_size = 100, last_row = 200, last_column = 200, plot = True)

## Step 6: Ensure array and particles are properly captured

In [ ]:
minr = 30
maxr = 330
minc = 44
maxc = 344
ref_bbox = (minr,minc,maxr,maxc)

### Check X

In [ ]:
dx_offset = 2
stagger_x = False
stagger_x_frequency = 2

checking = True
num_rows = 1
num_cols = 25
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = 0,
                                                         stagger_x = stagger_x, stagger_y = False, stagger_y_frequency = 2, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

### Check Y

In [ ]:
dy_offset = 1
stagger_y = True
stagger_y_frequency = 5

checking = True
num_rows = 25
num_cols = 1
article_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = dy_offset,
                                                         stagger_x = stagger_x, stagger_y = stagger_y, stagger_y_frequency = stagger_y_frequency, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

## Step 7: Apply all parameters and pull out particles from debris

In [ ]:
dx_offset = 2
stagger_x = False
stagger_x_frequency = 2

dy_offset = 1
stagger_y = True
stagger_y_frequency = 5

checking = False
erode_pixels = 2
num_rows = 25
num_cols = 25
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = dy_offset,
                                                          stagger_x = stagger_x, stagger_y = stagger_y, stagger_x_frequency = stagger_x_frequency, stagger_y_frequency = stagger_y_frequency,
                                                          erode_pixels = erode_pixels, checking = checking, check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

In [ ]:
caf.display_mask(particle_mask, 10, 10)

In [ ]:
caf.display_mask(particle_mask[0:360,20:360], 10, 10)

In [ ]:
caf.display_mask(debris_mask,10,10)
caf.display_mask(debris_mask[1200:1600, 1200:1600],10,10)

In [ ]:
caf.save_mask(particle_mask, image_path, '_0.png')
caf.save_mask(debris_mask, image_path,'_debris_0.png')

## Step 7: Identify bounds where particles are missing on the final wafer

### Step 7a: Load in cut wafer image and convert to a mask

In [ ]:
original_image_name = '0.2wt_no-field_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.2_0mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

In [ ]:
cut_mask = caf.cropped_image_to_mask(cropped_image, method="otsu")

### Step 7b: Rotate the Mask Roughly

In [ ]:
angle = 2
cut_mask_rotated = rotate(cut_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated,10,10)

### Step 7c: Perform Initial Crop

In [ ]:
ymin = 360
ymax = 3960
xmin = 455
xmax =3645
cut_mask = cut_mask_rotated[ymin:ymax, xmin:xmax]
caf.display_mask(cut_mask,10,10)

### Step 7d: Finely Rotate the Mask

In [ ]:
rot_angle_mask = cut_mask[1085:,:-420]
caf.display_mask(rot_angle_mask,6,6)

In [ ]:
rot_angle_mask = ~rot_angle_mask
rot_angle_cut_mask = morphology.remove_small_objects(rot_angle_mask, min_size= 200) #Remove noise to keep it from interfering
caf.display_mask(rot_angle_cut_mask, 10,10)
rot_angle_cut_mask, cut_angle = caf.rotate_mask_until_balanced(rot_angle_cut_mask, angle_step=0.2, tolerance=0.0005, min_step=0.001, max_angle=2, search_rows = 10, search_columns = 50)
print(cut_angle)
caf.display_mask(rot_angle_cut_mask, 10,10)

In [ ]:
angle = 0.1
cut_mask_rotated = rotate(cut_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated, 10, 10)

### Step 7e: Identify the appropriate bounds in particle mask (roughly)

In [ ]:
plus_x = 2350
plus_y =2750
match_mask = particle_mask[-np.shape(cut_mask_rotated)[0]-plus_y:, :np.shape(cut_mask_rotated)[1]+plus_x]
caf.display_mask(match_mask,10, 10)

In [ ]:
modify_rows_left = 2
modify_rows_right = 10
modify_cols_top = 60
modify_cols_bottom = 0
cut_mask2 =  caf.match_masks(~cut_mask_rotated, match_mask, modify_rows_left, modify_rows_right, modify_cols_top, modify_cols_bottom)

In [ ]:
caf.display_mask(cut_mask2, 10, 10)

### Step 7f: Add rows and columns to match with the particle mask

In [ ]:
cut_mask3 = caf.add_rows_to_match(~cut_mask2, particle_mask)

### Step 7g: Clean the mask to separate the area that has no particles

In [ ]:
caf.display_mask(cut_mask3,10,10)

In [ ]:
cut_mask4 = caf.isolate_empty_space2(cut_mask3, remove_particle_size = 100000, small_object_size = 2000, max_column_gap1 = 100, max_row_gap = 200, max_filled_col_gap = 500,
                        max_filled_row_gap = 500, max_column_gap2 = 50, enhance_large_gap_size =200000, large_hole_threshold = 2000000, plot = True)

## Step 8: Filter particle mask using blank space from cut mask

In [ ]:
caf.display_mask(particle_mask,10,10)

In [ ]:
caf.display_mask(cut_mask3,10,10)

In [ ]:
original_image_name = '0.2wt_no-field_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.2_0mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

In [ ]:
filtered_particle_mask = particle_mask & cut_mask4
filtered_debris_mask = debris_mask & cut_mask4
caf.display_mask(filtered_particle_mask,10,10)
filtered_particle_mask_cropped = filtered_particle_mask[:5650,:5300]
filtered_debris_mask_cropped = filtered_debris_mask[:5650,:5300]
caf.display_mask(filtered_particle_mask_cropped,10,10)
caf.display_mask(filtered_debris_mask_cropped,10,10)

In [ ]:
save_mask(filtered_particle_mask_cropped,image_path,'_3.png')

# Identifying Chains in the Mask

## Below is a limited sample of the code that identifies all of the different chains within the mask. The full code can be run in "Run_chain_analysis.py"

In [ ]:
original_image_name = '55-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.2_0mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask_name = '55-connectors_cleaned_mask.png'
filtered_particle_mask_name = os.path.join(os.getcwd(), '0.2_0mT', mask_name)
filtered_particle_mask = Image.open(filtered_particle_mask_name).convert('L')
filtered_particle_mask = np.array(filtered_particle_mask) == 255
caf.display_mask(filtered_particle_mask)

In [ ]:
particle_bounds = (800,1200,800,1200)
region_counter, chain_mask, particle_region_masks = caf.label_mask_16(filtered_particle_mask, original_image, particle_bounds, disk_size=0, connectivity=1,
                                                                    branch_length_fraction=0.007, global_min_branch_length=2, min_region_size=200, debug_plots = False,
                                                                    prune=True, prune_branch_length=5, max_hole_size=20, vertical_prune_length = 4)

## Load Completed Labelling for a Wafer

In [ ]:
mask_name = '55-labels.csv'
wafer_path = os.path.join(os.getcwd(), '0.2_0mT', mask_name)
full_chain_mask = np.loadtxt(wafer_path, delimiter=",")
chain_mask = full_chain_mask[800:1600, 800:1600]
unique_labels = np.unique(chain_mask)
unique_labels = unique_labels[unique_labels != 0]
num_labels = len(unique_labels)

# Generate N visually distinct colors using HSV space
hues = np.linspace(0, 1, num_labels, endpoint=False)
np.random.seed(np.random.randint(0,100))  # Optional: fix randomness
np.random.shuffle(hues)  # Shuffle to avoid nearby labels looking similar
colors = hsv_to_rgb(np.stack([hues, np.ones_like(hues)*0.65, np.ones_like(hues)*0.95], axis=1))

# Create a mapping from label to color
label_to_color = {label: np.append(colors[i], 1.0) for i, label in enumerate(unique_labels)}  # RGBA

# Create the overlay image
overlay_img = np.zeros((*chain_mask.shape, 4), dtype=float)
for label, rgba in label_to_color.items():
    overlay_img[chain_mask == label] = rgba

# Add black boundaries
boundaries = segmentation.find_boundaries(chain_mask.astype(np.int32), mode='outer')
overlay_img[boundaries] = [0, 0, 0, 1]

# Display the result
fig, ax1 = plt.subplots(1, 1, figsize=(20, 10))
ax1.imshow(overlay_img)
ax1.set_title('Watershed-filled branches (globally unique colors) with black outlines')
ax1.axis('off')
plt.show()

# Analyzing Chain Data

### Analyze the properties of a small subset of the identified chains and debris in the image

In [ ]:
mask_name = '55-connectors_cleaned_mask_debris.png'
debris_mask_name = os.path.join(os.getcwd(), '0.2_0mT', mask_name)
debris_mask = Image.open(debris_mask_name).convert('L')
debris_mask = np.array(debris_mask) == 255
debris_mask = debris_mask[0:500,0:500] #Look at a small number of particles at once

In [ ]:
df = caf.analyze_clusters(chain_mask, 1, angle_offset = None, fixed_angle = 0, plot = False, print_statement = False)

In [ ]:
ex_mask = np.where(chain_mask == 8839, 5, 0)
caf.display_mask(ex_mask, 10, 10)
ex_df = caf.analyze_clusters(ex_mask, 1, angle_offset = None, fixed_angle = 0, plot = True, print_statement = True)